In [1]:

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image, ImageFile
import os
import glob
from tqdm import tqdm
import json
from datetime import datetime
import warnings
import logging
from pathlib import Path
import random

# Setup
ImageFile.LOAD_TRUNCATED_IMAGES = True
warnings.filterwarnings('ignore')
plt.switch_backend('Agg')

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

# Import SSIM
try:
    from skimage.metrics import structural_similarity as ssim
except ImportError:
    import subprocess
    subprocess.check_call(["pip", "install", "scikit-image"])
    from skimage.metrics import structural_similarity as ssim

# ============================================================================
# CONFIGURATION
# ============================================================================

class Configuration:
    """Centralized configuration management"""
    
    def __init__(
        self,
        data_root="D:/Jafar/New Low-dose/CH/",
        output_folder="D:/Jafar/New Low-dose/CH/2(50%)/Generated/",
        image_size=256,
        batch_size=4,
        num_epochs=150,
        learning_rate=0.0002,
        device='auto'
    ):
        # Paths
        self.data_root = data_root
        self.input_folder = os.path.join(data_root, "2(50%)/train")
        self.target_folder = os.path.join(data_root, "Normal/train")
        self.output_folder = output_folder
        
        # Training parameters
        self.image_size = image_size
        self.batch_size = batch_size
        self.num_epochs = num_epochs
        self.learning_rate = learning_rate
        
        # Data split ratios (must sum to ~1.0)
        self.train_ratio = 0.70
        self.val_ratio = 0.15
        self.test_ratio = 0.15
        
        # Other settings
        self.random_seed = 42
        self.save_frequency = 10
        self.num_workers = 0
        
        # Device setup
        self.device = self._configure_device(device)
        
        # Create experiment directory
        self.experiment_dir = self._create_experiment_dir()
        
        self._log_configuration()
    
    def _configure_device(self, device):
        """Configure computing device"""
        if device == 'auto':
            if torch.cuda.is_available():
                device = 'cuda'
                logger.info(f"Using GPU: {torch.cuda.get_device_name(0)}")
            else:
                device = 'cpu'
                logger.info("Using CPU")
        return device
    
    def _create_experiment_dir(self):
        """Create directory structure for experiment"""
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        exp_dir = os.path.join(
            self.output_folder,
            f"experiment_{timestamp}"
        )
        
        for subdir in ['checkpoints', 'samples', 'logs', 'metrics']:
            os.makedirs(os.path.join(exp_dir, subdir), exist_ok=True)
        
        return exp_dir
    
    def _log_configuration(self):
        """Log configuration details"""
        logger.info("="*60)
        logger.info("CONFIGURATION")
        logger.info("="*60)
        logger.info(f"Data Root: {self.data_root}")
        logger.info(f"Input Folder: {self.input_folder}")
        logger.info(f"Target Folder: {self.target_folder}")
        logger.info(f"Output: {self.experiment_dir}")
        logger.info(f"Image Size: {self.image_size}x{self.image_size}")
        logger.info(f"Batch Size: {self.batch_size}")
        logger.info(f"Epochs: {self.num_epochs}")
        logger.info(f"Learning Rate: {self.learning_rate}")
        logger.info(f"Device: {self.device}")
        logger.info(f"Split: Train={self.train_ratio:.0%}, Val={self.val_ratio:.0%}, Test={self.test_ratio:.0%}")
        logger.info("="*60)

# ============================================================================
# LOSS FUNCTIONS
# ============================================================================

class SSIMLoss(nn.Module):
    """SSIM Loss implementation for image quality"""
    
    def __init__(self, window_size=11, data_range=2.0):
        super().__init__()
        self.window_size = window_size
        self.data_range = data_range
        self.channel = 3
        self.window = self._create_window(window_size, self.channel)
    
    def _gaussian(self, window_size, sigma):
        """Create Gaussian kernel"""
        gauss = torch.Tensor([
            torch.exp(torch.tensor(-(x - window_size // 2) ** 2 / (2 * sigma ** 2)))
            for x in range(window_size)
        ])
        return gauss / gauss.sum()
    
    def _create_window(self, window_size, channel):
        """Create 2D Gaussian window"""
        _1D_window = self._gaussian(window_size, 1.5).unsqueeze(1)
        _2D_window = _1D_window.mm(_1D_window.t()).float().unsqueeze(0).unsqueeze(0)
        window = _2D_window.expand(channel, 1, window_size, window_size).contiguous()
        return window
    
    def _compute_ssim(self, img1, img2, window, window_size, channel):
        """Compute SSIM between two images"""
        mu1 = F.conv2d(img1, window, padding=window_size // 2, groups=channel)
        mu2 = F.conv2d(img2, window, padding=window_size // 2, groups=channel)
        
        mu1_sq = mu1.pow(2)
        mu2_sq = mu2.pow(2)
        mu1_mu2 = mu1 * mu2
        
        sigma1_sq = F.conv2d(img1 * img1, window, padding=window_size // 2, groups=channel) - mu1_sq
        sigma2_sq = F.conv2d(img2 * img2, window, padding=window_size // 2, groups=channel) - mu2_sq
        sigma12 = F.conv2d(img1 * img2, window, padding=window_size // 2, groups=channel) - mu1_mu2
        
        C1 = (0.01 * self.data_range) ** 2
        C2 = (0.03 * self.data_range) ** 2
        
        ssim_map = ((2 * mu1_mu2 + C1) * (2 * sigma12 + C2)) / \
                   ((mu1_sq + mu2_sq + C1) * (sigma1_sq + sigma2_sq + C2))
        
        return ssim_map.mean()
    
    def forward(self, img1, img2):
        """Forward pass"""
        (_, channel, _, _) = img1.size()
        
        if channel == self.channel and self.window.data.type() == img1.data.type():
            window = self.window
        else:
            window = self._create_window(self.window_size, channel)
            if img1.is_cuda:
                window = window.cuda(img1.get_device())
            window = window.type_as(img1)
            self.window = window
            self.channel = channel
        
        return 1 - self._compute_ssim(img1, img2, window, self.window_size, channel)


class CombinedLoss(nn.Module):
    """Combined loss function with L1, SSIM, and adversarial components"""
    
    def __init__(self, lambda_l1=50.0, lambda_ssim=30.0, lambda_adv=1.0):
        super().__init__()
        self.l1_loss = nn.L1Loss()
        self.mse_loss = nn.MSELoss()
        self.ssim_loss = SSIMLoss()
        
        self.lambda_l1 = lambda_l1
        self.lambda_ssim = lambda_ssim
        self.lambda_adv = lambda_adv
    
    def generator_loss(self, generated, target, disc_fake_output):
        """Calculate generator loss"""
        l1 = self.l1_loss(generated, target)
        ssim = self.ssim_loss(generated, target)
        adversarial = self.mse_loss(disc_fake_output, torch.ones_like(disc_fake_output))
        
        total = (self.lambda_l1 * l1 + 
                self.lambda_ssim * ssim + 
                self.lambda_adv * adversarial)
        
        return {
            'total': total,
            'l1': l1,
            'ssim': ssim,
            'adversarial': adversarial
        }
    
    def discriminator_loss(self, disc_real_output, disc_fake_output):
        """Calculate discriminator loss"""
        real_loss = self.mse_loss(disc_real_output, torch.ones_like(disc_real_output))
        fake_loss = self.mse_loss(disc_fake_output, torch.zeros_like(disc_fake_output))
        return (real_loss + fake_loss) * 0.5

# ============================================================================
# NETWORK ARCHITECTURES
# ============================================================================

class UNetGenerator(nn.Module):
    """U-Net Generator with skip connections"""
    
    def __init__(self, input_channels=3, output_channels=3, base_filters=64):
        super().__init__()
        
        # Encoder layers
        self.enc1 = self._encoder_block(input_channels, base_filters, normalize=False)
        self.enc2 = self._encoder_block(base_filters, base_filters * 2)
        self.enc3 = self._encoder_block(base_filters * 2, base_filters * 4)
        self.enc4 = self._encoder_block(base_filters * 4, base_filters * 8)
        self.enc5 = self._encoder_block(base_filters * 8, base_filters * 8)
        self.enc6 = self._encoder_block(base_filters * 8, base_filters * 8)
        
        # Bottleneck
        self.bottleneck = nn.Sequential(
            nn.Conv2d(base_filters * 8, base_filters * 8, 4, 2, 1),
            nn.ReLU(True),
            nn.ConvTranspose2d(base_filters * 8, base_filters * 8, 4, 2, 1),
            nn.InstanceNorm2d(base_filters * 8),
            nn.ReLU(True)
        )
        
        # Decoder layers
        self.dec6 = self._decoder_block(base_filters * 16, base_filters * 8, dropout=True)
        self.dec5 = self._decoder_block(base_filters * 16, base_filters * 8, dropout=True)
        self.dec4 = self._decoder_block(base_filters * 16, base_filters * 4)
        self.dec3 = self._decoder_block(base_filters * 8, base_filters * 2)
        self.dec2 = self._decoder_block(base_filters * 4, base_filters)
        
        # Final layer
        self.final = nn.Sequential(
            nn.ConvTranspose2d(base_filters * 2, output_channels, 4, 2, 1),
            nn.Tanh()
        )
        
        self._initialize_weights()
    
    def _encoder_block(self, in_channels, out_channels, normalize=True):
        """Create encoder block"""
        layers = [
            nn.Conv2d(in_channels, out_channels, 4, 2, 1),
        ]
        if normalize:
            layers.append(nn.InstanceNorm2d(out_channels))
        layers.append(nn.LeakyReLU(0.2, True))
        return nn.Sequential(*layers)
    
    def _decoder_block(self, in_channels, out_channels, dropout=False):
        """Create decoder block"""
        layers = [
            nn.ConvTranspose2d(in_channels, out_channels, 4, 2, 1),
            nn.InstanceNorm2d(out_channels),
        ]
        if dropout:
            layers.append(nn.Dropout2d(0.5))
        layers.append(nn.ReLU(True))
        return nn.Sequential(*layers)
    
    def _initialize_weights(self):
        """Initialize network weights"""
        for m in self.modules():
            if isinstance(m, (nn.Conv2d, nn.ConvTranspose2d)):
                nn.init.normal_(m.weight, 0.0, 0.02)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, nn.InstanceNorm2d):
                if m.weight is not None:
                    nn.init.normal_(m.weight, 1.0, 0.02)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
    
    def forward(self, x):
        """Forward pass with skip connections"""
        # Encoder
        e1 = self.enc1(x)
        e2 = self.enc2(e1)
        e3 = self.enc3(e2)
        e4 = self.enc4(e3)
        e5 = self.enc5(e4)
        e6 = self.enc6(e5)
        
        # Bottleneck
        b = self.bottleneck(e6)
        
        # Decoder with skip connections
        d6 = self.dec6(torch.cat([b, e6], 1))
        d5 = self.dec5(torch.cat([d6, e5], 1))
        d4 = self.dec4(torch.cat([d5, e4], 1))
        d3 = self.dec3(torch.cat([d4, e3], 1))
        d2 = self.dec2(torch.cat([d3, e2], 1))
        
        return self.final(torch.cat([d2, e1], 1))


class PatchGANDiscriminator(nn.Module):
    """PatchGAN Discriminator"""
    
    def __init__(self, input_channels=6, base_filters=64):
        super().__init__()
        
        self.model = nn.Sequential(
            # Layer 1: No normalization
            nn.Conv2d(input_channels, base_filters, 4, 2, 1),
            nn.LeakyReLU(0.2, True),
            
            # Layer 2
            nn.Conv2d(base_filters, base_filters * 2, 4, 2, 1),
            nn.InstanceNorm2d(base_filters * 2),
            nn.LeakyReLU(0.2, True),
            
            # Layer 3
            nn.Conv2d(base_filters * 2, base_filters * 4, 4, 2, 1),
            nn.InstanceNorm2d(base_filters * 4),
            nn.LeakyReLU(0.2, True),
            
            # Layer 4
            nn.Conv2d(base_filters * 4, base_filters * 8, 4, 1, 1),
            nn.InstanceNorm2d(base_filters * 8),
            nn.LeakyReLU(0.2, True),
            
            # Output layer
            nn.Conv2d(base_filters * 8, 1, 4, 1, 1)
        )
        
        self._initialize_weights()
    
    def _initialize_weights(self):
        """Initialize network weights"""
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.normal_(m.weight, 0.0, 0.02)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, nn.InstanceNorm2d):
                if m.weight is not None:
                    nn.init.normal_(m.weight, 1.0, 0.02)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
    
    def forward(self, input_img, target_img):
        """Forward pass"""
        x = torch.cat([input_img, target_img], dim=1)
        return self.model(x)

# ============================================================================
# METRICS
# ============================================================================

class MetricsTracker:
    """Track and calculate image quality metrics"""
    
    def __init__(self, device):
        self.device = device
        self.best_metrics = self._initialize_best_metrics()
    
    def _initialize_best_metrics(self):
        """Initialize best metrics tracking"""
        return {
            'psnr': {'value': 0.0, 'epoch': 0, 'higher_better': True},
            'ssim': {'value': 0.0, 'epoch': 0, 'higher_better': True},
            'mae': {'value': float('inf'), 'epoch': 0, 'higher_better': False},
            'mse': {'value': float('inf'), 'epoch': 0, 'higher_better': False},
            'rmse': {'value': float('inf'), 'epoch': 0, 'higher_better': False},
            'nrmse': {'value': float('inf'), 'epoch': 0, 'higher_better': False},
            'snr': {'value': 0.0, 'epoch': 0, 'higher_better': True}
        }
    
    def calculate_psnr(self, img1, img2, max_val=2.0):
        """Calculate Peak Signal-to-Noise Ratio"""
        mse = torch.mean((img1 - img2) ** 2)
        if mse == 0:
            return torch.tensor(100.0)
        return 20 * torch.log10(max_val / torch.sqrt(mse))
    
    def calculate_rmse(self, img1, img2):
        """Calculate Root Mean Square Error"""
        mse = torch.mean((img1 - img2) ** 2)
        return torch.sqrt(mse)
    
    def calculate_nrmse(self, img1, img2):
        """Calculate Normalized Root Mean Square Error"""
        rmse = self.calculate_rmse(img1, img2)
        target_range = torch.max(img2) - torch.min(img2)
        if target_range == 0:
            return torch.tensor(0.0)
        return rmse / target_range
    
    def calculate_snr(self, img1, img2):
        """Calculate Signal-to-Noise Ratio"""
        signal_power = torch.mean(img2 ** 2)
        noise_power = torch.mean((img1 - img2) ** 2)
        if noise_power == 0:
            return torch.tensor(100.0)
        return 10 * torch.log10(signal_power / noise_power)
    
    def calculate_ssim(self, img1, img2):
        """Calculate SSIM using scikit-image"""
        try:
            img1_np = img1.detach().cpu().numpy()
            img2_np = img2.detach().cpu().numpy()
            
            ssim_values = []
            for i in range(img1_np.shape[0]):
                im1 = np.transpose(img1_np[i], (1, 2, 0))
                im2 = np.transpose(img2_np[i], (1, 2, 0))
                
                # Normalize to [0, 1]
                im1 = np.clip((im1 + 1) / 2, 0, 1)
                im2 = np.clip((im2 + 1) / 2, 0, 1)
                
                ssim_val = ssim(im1, im2, multichannel=True, 
                               data_range=1.0, channel_axis=2)
                ssim_values.append(ssim_val)
            
            return np.mean(ssim_values)
        except Exception as e:
            logger.warning(f"SSIM calculation failed: {e}")
            return 0.5
    
    def compute_all_metrics(self, generated, target):
        """Compute all metrics at once"""
        with torch.no_grad():
            try:
                metrics = {
                    'psnr': self.calculate_psnr(generated, target).item(),
                    'ssim': self.calculate_ssim(generated, target),
                    'mae': F.l1_loss(generated, target).item(),
                    'mse': F.mse_loss(generated, target).item(),
                    'rmse': self.calculate_rmse(generated, target).item(),
                    'nrmse': self.calculate_nrmse(generated, target).item(),
                    'snr': self.calculate_snr(generated, target).item()
                }
            except Exception as e:
                logger.warning(f"Metrics calculation error: {e}")
                metrics = {
                    'psnr': 0.0, 'ssim': 0.0, 'mae': 1.0, 'mse': 1.0,
                    'rmse': 1.0, 'nrmse': 1.0, 'snr': 0.0
                }
        
        return metrics
    
    def update_best(self, metrics, epoch):
        """Update best metrics"""
        for metric_name, value in metrics.items():
            if metric_name in self.best_metrics:
                current = self.best_metrics[metric_name]
                is_better = (value > current['value']) if current['higher_better'] \
                           else (value < current['value'])
                
                if is_better:
                    self.best_metrics[metric_name]['value'] = value
                    self.best_metrics[metric_name]['epoch'] = epoch
    
    def print_best_metrics(self):
        """Print best metrics summary"""
        print("\n" + "="*80)
        print("BEST METRICS ACHIEVED")
        print("="*80)
        
        print("\nQuality Metrics (Higher is Better):")
        print("-" * 50)
        for name in ['psnr', 'ssim', 'snr']:
            data = self.best_metrics[name]
            unit = ' dB' if name in ['psnr', 'snr'] else ''
            print(f"  {name.upper():8}: {data['value']:8.4f}{unit:4}  (Epoch {data['epoch']})")
        
        print("\nError Metrics (Lower is Better):")
        print("-" * 50)
        for name in ['mae', 'mse', 'rmse', 'nrmse']:
            data = self.best_metrics[name]
            print(f"  {name.upper():8}: {data['value']:8.6f}      (Epoch {data['epoch']})")
        
        print("="*80)
    
    def save_to_file(self, filepath):
        """Save best metrics to JSON"""
        try:
            with open(filepath, 'w') as f:
                json.dump(self.best_metrics, f, indent=4)
            logger.info(f"Best metrics saved to: {filepath}")
        except Exception as e:
            logger.error(f"Failed to save metrics: {e}")

# ============================================================================
# DATA HANDLING
# ============================================================================

class MedicalImageDataset(Dataset):
    """Dataset for paired medical images"""
    
    def __init__(self, image_pairs, config, augment=True):
        self.image_pairs = image_pairs
        self.config = config
        self.augment = augment
        
        self.transform = self._build_transform()
    
    def _build_transform(self):
        """Build image transformation pipeline"""
        transforms_list = [
            transforms.Resize((self.config.image_size, self.config.image_size)),
        ]
        
        if self.augment:
            transforms_list.append(transforms.RandomHorizontalFlip(0.5))
        
        transforms_list.extend([
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
        ])
        
        return transforms.Compose(transforms_list)
    
    def __len__(self):
        return len(self.image_pairs)
    
    def __getitem__(self, idx):
        try:
            input_path, target_path = self.image_pairs[idx]
            
            input_img = Image.open(input_path).convert('RGB')
            target_img = Image.open(target_path).convert('RGB')
            
            # Synchronized augmentation
            if self.augment:
                seed = random.randint(0, 2**32 - 1)
                random.seed(seed)
                torch.manual_seed(seed)
                input_tensor = self.transform(input_img)
                
                random.seed(seed)
                torch.manual_seed(seed)
                target_tensor = self.transform(target_img)
            else:
                input_tensor = self.transform(input_img)
                target_tensor = self.transform(target_img)
            
            return {
                'input': input_tensor,
                'target': target_tensor,
                'filename': Path(input_path).stem
            }
        
        except Exception as e:
            logger.warning(f"Error loading image pair {idx}: {e}")
            # Return dummy tensors
            dummy = torch.zeros(3, self.config.image_size, self.config.image_size)
            return {
                'input': dummy,
                'target': dummy,
                'filename': 'error'
            }


class DatasetManager:
    """Manage dataset creation and splitting"""
    
    def __init__(self, config):
        self.config = config
        self.image_pairs = []
    
    def prepare_datasets(self):
        """Find pairs and create train/val/test splits"""
        logger.info("Searching for image pairs...")
        self._find_pairs()
        
        if len(self.image_pairs) == 0:
            raise ValueError("No valid image pairs found!")
        
        logger.info(f"Found {len(self.image_pairs)} image pairs")
        
        # Split data
        train_pairs, val_pairs, test_pairs = self._split_data()
        
        # Create datasets
        train_dataset = MedicalImageDataset(train_pairs, self.config, augment=True)
        val_dataset = MedicalImageDataset(val_pairs, self.config, augment=False) if val_pairs else None
        test_dataset = MedicalImageDataset(test_pairs, self.config, augment=False) if test_pairs else None
        
        return train_dataset, val_dataset, test_dataset
    
    def _find_pairs(self):
        """Find matching input-target image pairs"""
        extensions = ['*.png', '*.jpg', '*.jpeg', '*.bmp', '*.tiff']
        
        input_files = []
        for ext in extensions:
            input_files.extend(glob.glob(
                os.path.join(self.config.input_folder, ext)
            ))
            input_files.extend(glob.glob(
                os.path.join(self.config.input_folder, ext.upper())
            ))
        
        valid_pairs = []
        for input_path in tqdm(input_files, desc="Validating pairs"):
            input_name = Path(input_path).stem
            
            # Search for matching target
            target_path = None
            for ext in ['.png', '.jpg', '.jpeg', '.bmp', '.tiff']:
                for case_ext in [ext, ext.upper()]:
                    candidate = os.path.join(
                        self.config.target_folder,
                        input_name + case_ext
                    )
                    if os.path.exists(candidate):
                        target_path = candidate
                        break
                if target_path:
                    break
            
            if target_path and self._validate_pair(input_path, target_path):
                valid_pairs.append((input_path, target_path))
        
        self.image_pairs = valid_pairs
    
    def _validate_pair(self, input_path, target_path):
        """Validate that both images can be loaded"""
        try:
            Image.open(input_path).convert('RGB')
            Image.open(target_path).convert('RGB')
            return True
        except:
            return False
    
    def _split_data(self):
        """Split data into train/val/test sets"""
        random.seed(self.config.random_seed)
        random.shuffle(self.image_pairs)
        
        total = len(self.image_pairs)
        train_size = int(total * self.config.train_ratio)
        val_size = int(total * self.config.val_ratio)
        
        train_pairs = self.image_pairs[:train_size]
        val_pairs = self.image_pairs[train_size:train_size + val_size]
        test_pairs = self.image_pairs[train_size + val_size:]
        
        logger.info(f"Data split - Train: {len(train_pairs)}, "
                   f"Val: {len(val_pairs)}, Test: {len(test_pairs)}")
        
        return train_pairs, val_pairs, test_pairs

# ============================================================================
# TRAINING
# ============================================================================

class Trainer:
    """Main training orchestrator"""
    
    def __init__(self, config):
        self.config = config
        
        # Initialize models
        self.generator = UNetGenerator().to(config.device)
        self.discriminator = PatchGANDiscriminator().to(config.device)
        
        # Optimizers
        self.g_optimizer = torch.optim.Adam(
            self.generator.parameters(),
            lr=config.learning_rate,
            betas=(0.5, 0.999)
        )
        self.d_optimizer = torch.optim.Adam(
            self.discriminator.parameters(),
            lr=config.learning_rate,
            betas=(0.5, 0.999)
        )
        
        # Loss and metrics
        self.criterion = CombinedLoss()
        self.metrics_tracker = MetricsTracker(config.device)
        
        # Training history
        self.history = {
            'train_g_loss': [], 'train_d_loss': [],
            'train_psnr': [], 'train_ssim': [], 'train_mae': [],
            'train_mse': [], 'train_rmse': [], 'train_nrmse': [], 'train_snr': [],
            'val_g_loss': [], 'val_d_loss': [],
            'val_psnr': [], 'val_ssim': [], 'val_mae': [],
            'val_mse': [], 'val_rmse': [], 'val_nrmse': [], 'val_snr': []
        }
        
        self.best_val_psnr = 0.0
        self.best_epoch = 0
        
        logger.info("Trainer initialized successfully")
    
    def train_epoch(self, dataloader, epoch):
        """Train for one epoch"""
        self.generator.train()
        self.discriminator.train()
        
        epoch_losses = {'g_loss': 0, 'd_loss': 0}
        epoch_metrics = {
            'psnr': 0, 'ssim': 0, 'mae': 0, 'mse': 0,
            'rmse': 0, 'nrmse': 0, 'snr': 0
        }
        num_batches = 0
        
        pbar = tqdm(dataloader, desc=f'Epoch {epoch}/{self.config.num_epochs}')
        
        for batch in pbar:
            input_img = batch['input'].to(self.config.device)
            target_img = batch['target'].to(self.config.device)
            
            if input_img.shape != target_img.shape:
                continue
            
            # ============ Train Discriminator ============
            self.d_optimizer.zero_grad()
            
            with torch.no_grad():
                fake_img = self.generator(input_img)
            
            real_pred = self.discriminator(input_img, target_img)
            fake_pred = self.discriminator(input_img, fake_img.detach())
            
            d_loss = self.criterion.discriminator_loss(real_pred, fake_pred)
            d_loss.backward()
            self.d_optimizer.step()
            
            # ============ Train Generator ============
            self.g_optimizer.zero_grad()
            
            fake_img = self.generator(input_img)
            fake_pred = self.discriminator(input_img, fake_img)
            
            g_losses = self.criterion.generator_loss(fake_img, target_img, fake_pred)
            g_loss = g_losses['total']
            
            g_loss.backward()
            self.g_optimizer.step()
            
            # ============ Calculate Metrics ============
            metrics = self.metrics_tracker.compute_all_metrics(fake_img, target_img)
            
            # Update tracking
            epoch_losses['g_loss'] += g_loss.item()
            epoch_losses['d_loss'] += d_loss.item()
            for key in epoch_metrics:
                epoch_metrics[key] += metrics[key]
            num_batches += 1
            
            # Update progress bar
            pbar.set_postfix({
                'G': f'{g_loss.item():.4f}',
                'D': f'{d_loss.item():.4f}',
                'PSNR': f'{metrics["psnr"]:.2f}',
                'SSIM': f'{metrics["ssim"]:.3f}'
            })
        
        if num_batches == 0:
            raise RuntimeError("No valid batches processed!")
        
        # Average all metrics
        for key in epoch_losses:
            epoch_losses[key] /= num_batches
        for key in epoch_metrics:
            epoch_metrics[key] /= num_batches
        
        return {**epoch_losses, **epoch_metrics}
    
    def validate(self, dataloader):
        """Validate the model"""
        self.generator.eval()
        self.discriminator.eval()
        
        val_losses = {'g_loss': 0, 'd_loss': 0}
        val_metrics = {
            'psnr': 0, 'ssim': 0, 'mae': 0, 'mse': 0,
            'rmse': 0, 'nrmse': 0, 'snr': 0
        }
        num_batches = 0
        
        with torch.no_grad():
            for batch in dataloader:
                input_img = batch['input'].to(self.config.device)
                target_img = batch['target'].to(self.config.device)
                
                if input_img.shape != target_img.shape:
                    continue
                
                fake_img = self.generator(input_img)
                
                real_pred = self.discriminator(input_img, target_img)
                fake_pred = self.discriminator(input_img, fake_img)
                
                d_loss = self.criterion.discriminator_loss(real_pred, fake_pred)
                g_losses = self.criterion.generator_loss(fake_img, target_img, fake_pred)
                g_loss = g_losses['total']
                
                metrics = self.metrics_tracker.compute_all_metrics(fake_img, target_img)
                
                val_losses['g_loss'] += g_loss.item()
                val_losses['d_loss'] += d_loss.item()
                for key in val_metrics:
                    val_metrics[key] += metrics[key]
                num_batches += 1
        
        if num_batches == 0:
            return None
        
        for key in val_losses:
            val_losses[key] /= num_batches
        for key in val_metrics:
            val_metrics[key] /= num_batches
        
        return {**val_losses, **val_metrics}
    
    def save_checkpoint(self, epoch, is_best=False):
        """Save model checkpoint"""
        checkpoint = {
            'epoch': epoch,
            'generator': self.generator.state_dict(),
            'discriminator': self.discriminator.state_dict(),
            'g_optimizer': self.g_optimizer.state_dict(),
            'd_optimizer': self.d_optimizer.state_dict(),
            'history': self.history,
            'best_val_psnr': self.best_val_psnr,
            'best_epoch': self.best_epoch,
            'best_metrics': self.metrics_tracker.best_metrics
        }
        
        checkpoint_path = os.path.join(
            self.config.experiment_dir,
            'checkpoints',
            f'checkpoint_epoch_{epoch}.pth'
        )
        torch.save(checkpoint, checkpoint_path)
        
        if is_best:
            best_path = os.path.join(
                self.config.experiment_dir,
                'checkpoints',
                'best_model.pth'
            )
            torch.save(checkpoint, best_path)
            logger.info(f"Saved best model at epoch {epoch} (PSNR: {self.best_val_psnr:.2f}dB)")
    
    def save_sample_images(self, dataloader, epoch, num_samples=4):
        """Save sample images for visualization"""
        self.generator.eval()
        
        samples_dir = os.path.join(self.config.experiment_dir, 'samples')
        
        with torch.no_grad():
            for i, batch in enumerate(dataloader):
                if i >= num_samples:
                    break
                
                input_img = batch['input'][:1].to(self.config.device)
                target_img = batch['target'][:1].to(self.config.device)
                filename = batch['filename'][0]
                
                generated_img = self.generator(input_img)
                
                # Convert to numpy and rescale
                input_np = ((input_img[0].cpu().numpy().transpose(1, 2, 0) + 1) / 2).clip(0, 1)
                target_np = ((target_img[0].cpu().numpy().transpose(1, 2, 0) + 1) / 2).clip(0, 1)
                generated_np = ((generated_img[0].cpu().numpy().transpose(1, 2, 0) + 1) / 2).clip(0, 1)
                
                # Create comparison plot
                fig, axes = plt.subplots(1, 3, figsize=(15, 5))
                
                axes[0].imshow(input_np)
                axes[0].set_title('Input (Low Dose)')
                axes[0].axis('off')
                
                axes[1].imshow(generated_np)
                axes[1].set_title('Generated')
                axes[1].axis('off')
                
                axes[2].imshow(target_np)
                axes[2].set_title('Target (High Dose)')
                axes[2].axis('off')
                
                plt.tight_layout()
                
                save_path = os.path.join(
                    samples_dir,
                    f'epoch_{epoch}_sample_{i+1}_{filename}.png'
                )
                plt.savefig(save_path, dpi=150, bbox_inches='tight')
                plt.close()
        
        logger.info(f"Sample images saved for epoch {epoch}")

# ============================================================================
# MAIN TRAINING PIPELINE
# ============================================================================

def run_training(config):
    """Execute the complete training pipeline"""
    
    logger.info("="*80)
    logger.info("STARTING TRAINING PIPELINE")
    logger.info("="*80)
    
    # Prepare datasets
    dataset_manager = DatasetManager(config)
    train_dataset, val_dataset, test_dataset = dataset_manager.prepare_datasets()
    
    # Create dataloaders
    train_loader = DataLoader(
        train_dataset,
        batch_size=config.batch_size,
        shuffle=True,
        num_workers=config.num_workers,
        drop_last=True,
        pin_memory=False
    )
    
    val_loader = None
    if val_dataset is not None:
        val_loader = DataLoader(
            val_dataset,
            batch_size=config.batch_size,
            shuffle=False,
            num_workers=config.num_workers,
            pin_memory=False
        )
    
    logger.info(f"Training batches: {len(train_loader)}")
    logger.info(f"Validation batches: {len(val_loader) if val_loader else 0}")
    
    # Initialize trainer
    trainer = Trainer(config)
    
    # Training loop
    for epoch in range(1, config.num_epochs + 1):
        try:
            # Train
            train_results = trainer.train_epoch(train_loader, epoch)
            
            # Validate
            val_results = None
            if val_loader is not None:
                val_results = trainer.validate(val_loader)
            
            # Update history
            for key in ['g_loss', 'd_loss', 'psnr', 'ssim', 'mae', 'mse', 'rmse', 'nrmse', 'snr']:
                trainer.history[f'train_{key}'].append(train_results[key])
                if val_results:
                    trainer.history[f'val_{key}'].append(val_results[key])
            
            # Update best metrics
            trainer.metrics_tracker.update_best(train_results, epoch)
            if val_results:
                trainer.metrics_tracker.update_best(val_results, epoch)
            
            # Print results
            print(f"\nEpoch {epoch} Results:")
            print(f"  Train - G_Loss: {train_results['g_loss']:.4f}, "
                  f"D_Loss: {train_results['d_loss']:.4f}")
            print(f"  Train - PSNR: {train_results['psnr']:.2f}dB, "
                  f"SSIM: {train_results['ssim']:.4f}, "
                  f"RMSE: {train_results['rmse']:.6f}, "
                  f"NRMSE: {train_results['nrmse']:.6f}")
            
            if val_results:
                print(f"  Val   - G_Loss: {val_results['g_loss']:.4f}, "
                      f"D_Loss: {val_results['d_loss']:.4f}")
                print(f"  Val   - PSNR: {val_results['psnr']:.2f}dB, "
                      f"SSIM: {val_results['ssim']:.4f}, "
                      f"RMSE: {val_results['rmse']:.6f}, "
                      f"NRMSE: {val_results['nrmse']:.6f}")
            
            # Check for best model
            is_best = False
            current_psnr = val_results['psnr'] if val_results else train_results['psnr']
            if current_psnr > trainer.best_val_psnr:
                trainer.best_val_psnr = current_psnr
                trainer.best_epoch = epoch
                is_best = True
                print(f"\n*** NEW BEST MODEL - PSNR: {current_psnr:.2f}dB ***")
            
            # Save checkpoint
            if epoch % config.save_frequency == 0 or is_best:
                trainer.save_checkpoint(epoch, is_best)
            
            # Save sample images
            if epoch % (config.save_frequency * 2) == 0 and val_loader:
                trainer.save_sample_images(val_loader, epoch)
            
            # Print best metrics periodically
            if epoch % 20 == 0:
                trainer.metrics_tracker.print_best_metrics()
        
        except Exception as e:
            logger.error(f"Error in epoch {epoch}: {e}")
            if epoch < 5:
                raise
            continue
    
    # Training complete
    logger.info("="*80)
    logger.info("TRAINING COMPLETED")
    logger.info("="*80)
    logger.info(f"Best PSNR: {trainer.best_val_psnr:.2f}dB at epoch {trainer.best_epoch}")
    
    # Print and save best metrics
    trainer.metrics_tracker.print_best_metrics()
    
    metrics_path = os.path.join(config.experiment_dir, 'metrics', 'best_metrics.json')
    trainer.metrics_tracker.save_to_file(metrics_path)
    
    history_path = os.path.join(config.experiment_dir, 'metrics', 'training_history.json')
    with open(history_path, 'w') as f:
        json.dump(trainer.history, f, indent=4)
    
    logger.info(f"Results saved to: {config.experiment_dir}")
    
    return config.experiment_dir, test_dataset

# ============================================================================
# EVALUATION
# ============================================================================

def evaluate_test_set(experiment_dir, test_dataset, config):
    """Evaluate the best model on test set"""
    
    logger.info("="*80)
    logger.info("TEST SET EVALUATION")
    logger.info("="*80)
    
    # Load best model
    best_model_path = os.path.join(experiment_dir, 'checkpoints', 'best_model.pth')
    if not os.path.exists(best_model_path):
        logger.error("Best model not found!")
        return None
    
    checkpoint = torch.load(best_model_path, map_location=config.device)
    
    generator = UNetGenerator().to(config.device)
    generator.load_state_dict(checkpoint['generator'])
    generator.eval()
    
    # Create test dataloader
    test_loader = DataLoader(
        test_dataset,
        batch_size=1,
        shuffle=False,
        num_workers=0
    )
    
    metrics_tracker = MetricsTracker(config.device)
    
    test_metrics = {
        'psnr': 0, 'ssim': 0, 'mae': 0, 'mse': 0,
        'rmse': 0, 'nrmse': 0, 'snr': 0
    }
    num_samples = 0
    
    logger.info(f"Evaluating {len(test_loader)} test samples...")
    
    with torch.no_grad():
        for batch in tqdm(test_loader, desc="Testing"):
            input_img = batch['input'].to(config.device)
            target_img = batch['target'].to(config.device)
            
            generated_img = generator(input_img)
            metrics = metrics_tracker.compute_all_metrics(generated_img, target_img)
            
            for key in test_metrics:
                test_metrics[key] += metrics[key]
            num_samples += 1
    
    if num_samples == 0:
        logger.error("No valid test samples!")
        return None
    
    # Average metrics
    for key in test_metrics:
        test_metrics[key] /= num_samples
    
    # Print results
    print("\n" + "="*80)
    print(f"TEST SET RESULTS ({num_samples} samples)")
    print("="*80)
    print(f"  PSNR:  {test_metrics['psnr']:.4f} dB")
    print(f"  SSIM:  {test_metrics['ssim']:.6f}")
    print(f"  SNR:   {test_metrics['snr']:.4f} dB")
    print(f"  MAE:   {test_metrics['mae']:.8f}")
    print(f"  MSE:   {test_metrics['mse']:.8f}")
    print(f"  RMSE:  {test_metrics['rmse']:.8f}")
    print(f"  NRMSE: {test_metrics['nrmse']:.8f}")
    print("="*80)
    
    # Save results
    test_results_path = os.path.join(experiment_dir, 'metrics', 'test_results.json')
    with open(test_results_path, 'w') as f:
        json.dump(test_metrics, f, indent=4)
    
    logger.info(f"Test results saved to: {test_results_path}")
    
    return test_metrics

# ============================================================================
# VISUALIZATION
# ============================================================================

def create_training_plots(experiment_dir):
    """Create comprehensive training plots"""
    
    history_path = os.path.join(experiment_dir, 'metrics', 'training_history.json')
    
    if not os.path.exists(history_path):
        logger.warning("Training history not found!")
        return
    
    with open(history_path, 'r') as f:
        history = json.load(f)
    
    epochs = range(1, len(history['train_g_loss']) + 1)
    
    fig, axes = plt.subplots(3, 3, figsize=(18, 15))
    
    # Loss plots
    axes[0, 0].plot(epochs, history['train_g_loss'], label='Train G')
    axes[0, 0].plot(epochs, history['train_d_loss'], label='Train D')
    if len(history['val_g_loss']) > 0:
        axes[0, 0].plot(epochs, history['val_g_loss'], label='Val G')
        axes[0, 0].plot(epochs, history['val_d_loss'], label='Val D')
    axes[0, 0].set_title('Losses')
    axes[0, 0].set_xlabel('Epoch')
    axes[0, 0].set_ylabel('Loss')
    axes[0, 0].legend()
    axes[0, 0].grid(True)
    
    # PSNR
    axes[0, 1].plot(epochs, history['train_psnr'], label='Train')
    if len(history['val_psnr']) > 0:
        axes[0, 1].plot(epochs, history['val_psnr'], label='Val')
    axes[0, 1].set_title('PSNR')
    axes[0, 1].set_xlabel('Epoch')
    axes[0, 1].set_ylabel('PSNR (dB)')
    axes[0, 1].legend()
    axes[0, 1].grid(True)
    
    # SSIM
    axes[0, 2].plot(epochs, history['train_ssim'], label='Train')
    if len(history['val_ssim']) > 0:
        axes[0, 2].plot(epochs, history['val_ssim'], label='Val')
    axes[0, 2].set_title('SSIM')
    axes[0, 2].set_xlabel('Epoch')
    axes[0, 2].set_ylabel('SSIM')
    axes[0, 2].legend()
    axes[0, 2].grid(True)
    
    # MAE
    axes[1, 0].plot(epochs, history['train_mae'], label='Train')
    if len(history['val_mae']) > 0:
        axes[1, 0].plot(epochs, history['val_mae'], label='Val')
    axes[1, 0].set_title('MAE')
    axes[1, 0].set_xlabel('Epoch')
    axes[1, 0].set_ylabel('MAE')
    axes[1, 0].legend()
    axes[1, 0].grid(True)
    
    # MSE
    axes[1, 1].plot(epochs, history['train_mse'], label='Train')
    if len(history['val_mse']) > 0:
        axes[1, 1].plot(epochs, history['val_mse'], label='Val')
    axes[1, 1].set_title('MSE')
    axes[1, 1].set_xlabel('Epoch')
    axes[1, 1].set_ylabel('MSE')
    axes[1, 1].legend()
    axes[1, 1].grid(True)
    
    # RMSE
    axes[1, 2].plot(epochs, history['train_rmse'], label='Train')
    if len(history['val_rmse']) > 0:
        axes[1, 2].plot(epochs, history['val_rmse'], label='Val')
    axes[1, 2].set_title('RMSE')
    axes[1, 2].set_xlabel('Epoch')
    axes[1, 2].set_ylabel('RMSE')
    axes[1, 2].legend()
    axes[1, 2].grid(True)
    
    # NRMSE
    axes[2, 0].plot(epochs, history['train_nrmse'], label='Train')
    if len(history['val_nrmse']) > 0:
        axes[2, 0].plot(epochs, history['val_nrmse'], label='Val')
    axes[2, 0].set_title('NRMSE')
    axes[2, 0].set_xlabel('Epoch')
    axes[2, 0].set_ylabel('NRMSE')
    axes[2, 0].legend()
    axes[2, 0].grid(True)
    
    # SNR
    axes[2, 1].plot(epochs, history['train_snr'], label='Train')
    if len(history['val_snr']) > 0:
        axes[2, 1].plot(epochs, history['val_snr'], label='Val')
    axes[2, 1].set_title('SNR')
    axes[2, 1].set_xlabel('Epoch')
    axes[2, 1].set_ylabel('SNR (dB)')
    axes[2, 1].legend()
    axes[2, 1].grid(True)
    
    # Hide the last subplot
    axes[2, 2].axis('off')
    
    plt.tight_layout()
    
    plots_path = os.path.join(experiment_dir, 'training_plots.png')
    plt.savefig(plots_path, dpi=300, bbox_inches='tight')
    plt.close()
    
    logger.info(f"Training plots saved to: {plots_path}")

# ============================================================================
# MAIN EXECUTION
# ============================================================================

def main():
    """Main execution function"""
    
    print("\n" + "="*80)
    print("Enhanced Pix2Pix for Medical Image Translation")
    print("SSIM Loss + Comprehensive Metrics")
    print("="*80 + "\n")
    
    try:
        # Create configuration
        config = Configuration(
            data_root="D:/Jafar/New Low-dose/CH/",
            
            output_folder="D:/Jafar/New Low-dose/CH/2(50%)/Generated/",
            image_size=256,
            batch_size=4,
            num_epochs=150,
            learning_rate=0.0002,
            device='auto'
        )
        
        # Run training
        experiment_dir, test_dataset = run_training(config)
        
        # Create training plots
        logger.info("Creating training plots...")
        create_training_plots(experiment_dir)
        
        # Evaluate on test set
        if test_dataset is not None and len(test_dataset) > 0:
            logger.info("Evaluating on test set...")
            test_results = evaluate_test_set(experiment_dir, test_dataset, config)
        
        print("\n" + "="*80)
        print("TRAINING PIPELINE COMPLETED SUCCESSFULLY")
        print("="*80)
        print(f"Results saved to: {experiment_dir}")
        print("\nGenerated files:")
        print("  - best_metrics.json: Best metrics during training")
        print("  - training_history.json: Complete training history")
        print("  - training_plots.png: Visualization of all metrics")
        print("  - test_results.json: Final test evaluation")
        print("  - checkpoints/: Model checkpoints")
        print("  - samples/: Sample generated images")
        print("="*80 + "\n")
        
        return experiment_dir
    
    except Exception as e:
        logger.error(f"Training failed: {e}")
        import traceback
        traceback.print_exc()
        return None


if __name__ == "__main__":
    main()

2025-11-17 13:47:39,699 - INFO - Using GPU: NVIDIA GeForce RTX 4080
2025-11-17 13:47:39,701 - INFO - ============================================================
2025-11-17 13:47:39,701 - INFO - CONFIGURATION
2025-11-17 13:47:39,701 - INFO - ============================================================
2025-11-17 13:47:39,702 - INFO - Data Root: D:/Jafar/New Low-dose/CH/
2025-11-17 13:47:39,702 - INFO - Input Folder: D:/Jafar/New Low-dose/CH/2(50%)/train
2025-11-17 13:47:39,702 - INFO - Target Folder: D:/Jafar/New Low-dose/CH/Normal/train
2025-11-17 13:47:39,703 - INFO - Output: D:/Jafar/New Low-dose/CH/2(50%)/Generated/experiment_20251117_134739
2025-11-17 13:47:39,703 - INFO - Image Size: 256x256
2025-11-17 13:47:39,704 - INFO - Batch Size: 4
2025-11-17 13:47:39,704 - INFO - Epochs: 150
2025-11-17 13:47:39,704 - INFO - Learning Rate: 0.0002
2025-11-17 13:47:39,705 - INFO - Device: cuda
2025-11-17 13:47:39,705 - INFO - Split: Train=70%, Val=15%, Test=15%
2025-11-17 13:47:39,705 - INFO 


Enhanced Pix2Pix for Medical Image Translation
SSIM Loss + Comprehensive Metrics



Validating pairs: 100%|██████████| 5370/5370 [00:10<00:00, 535.09it/s]
2025-11-17 13:47:49,774 - INFO - Found 5370 image pairs
2025-11-17 13:47:49,778 - INFO - Data split - Train: 3758, Val: 805, Test: 807
2025-11-17 13:47:49,778 - INFO - Training batches: 939
2025-11-17 13:47:49,779 - INFO - Validation batches: 202
2025-11-17 13:47:50,286 - INFO - Trainer initialized successfully
Epoch 1/150: 100%|██████████| 939/939 [01:41<00:00,  9.25it/s, G=1.5085, D=0.1926, PSNR=41.28, SSIM=0.989]



Epoch 1 Results:
  Train - G_Loss: 3.5407, D_Loss: 0.2173
  Train - PSNR: 39.56dB, SSIM: 0.9644, RMSE: 0.043681, NRMSE: 0.027285
  Val   - G_Loss: 1.3562, D_Loss: 0.1094
  Val   - PSNR: 40.70dB, SSIM: 0.9911, RMSE: 0.018775, NRMSE: 0.011305

*** NEW BEST MODEL - PSNR: 40.70dB ***


2025-11-17 13:49:53,375 - INFO - Saved best model at epoch 1 (PSNR: 40.70dB)
Epoch 2/150: 100%|██████████| 939/939 [01:42<00:00,  9.20it/s, G=1.2143, D=0.0859, PSNR=45.87, SSIM=0.994]


...


Epoch 150 Results:
  Train - G_Loss: 1.1564, D_Loss: 0.0502
  Train - PSNR: 48.07dB, SSIM: 0.9956, RMSE: 0.008043, NRMSE: 0.005055
  Val   - G_Loss: 1.1518, D_Loss: 0.0526
  Val   - PSNR: 48.13dB, SSIM: 0.9957, RMSE: 0.008002, NRMSE: 0.004833


2025-11-17 18:54:33,023 - INFO - ================================================================================
2025-11-17 18:54:33,024 - INFO - TRAINING COMPLETED
2025-11-17 18:54:33,025 - INFO - ================================================================================
2025-11-17 18:54:33,025 - INFO - Best PSNR: 48.35dB at epoch 131
2025-11-17 18:54:33,027 - INFO - Best metrics saved to: D:/Jafar/New Low-dose/CH/2(50%)/Generated/experiment_20251117_134739\metrics\best_metrics.json
2025-11-17 18:54:33,034 - INFO - Results saved to: D:/Jafar/New Low-dose/CH/2(50%)/Generated/experiment_20251117_134739
2025-11-17 18:54:33,036 - INFO - Creating training plots...



BEST METRICS ACHIEVED

Quality Metrics (Higher is Better):
--------------------------------------------------
  PSNR    :  48.3461 dB   (Epoch 131)
  SSIM    :   0.9959      (Epoch 143)
  SNR     :  42.0944 dB   (Epoch 131)

Error Metrics (Lower is Better):
--------------------------------------------------
  MAE     : 0.002518      (Epoch 145)
  MSE     : 0.000064      (Epoch 131)
  RMSE    : 0.007805      (Epoch 131)
  NRMSE   : 0.004715      (Epoch 131)


2025-11-17 18:54:35,351 - INFO - Training plots saved to: D:/Jafar/New Low-dose/CH/2(50%)/Generated/experiment_20251117_134739\training_plots.png
2025-11-17 18:54:35,352 - INFO - Evaluating on test set...
2025-11-17 18:54:35,353 - INFO - ================================================================================
2025-11-17 18:54:35,353 - INFO - TEST SET EVALUATION
2025-11-17 18:54:35,354 - INFO - ================================================================================
2025-11-17 18:54:36,072 - INFO - Evaluating 807 test samples...
Testing: 100%|██████████| 807/807 [00:28<00:00, 28.80it/s]
2025-11-17 18:55:04,096 - INFO - Test results saved to: D:/Jafar/New Low-dose/CH/2(50%)/Generated/experiment_20251117_134739\metrics\test_results.json



TEST SET RESULTS (807 samples)
  PSNR:  49.0466 dB
  SSIM:  0.995684
  SNR:   42.7911 dB
  MAE:   0.00274436
  MSE:   0.00006355
  RMSE:  0.00745653
  NRMSE: 0.00804450

TRAINING PIPELINE COMPLETED SUCCESSFULLY
Results saved to: D:/Jafar/New Low-dose/CH/2(50%)/Generated/experiment_20251117_134739

Generated files:
  - best_metrics.json: Best metrics during training
  - training_history.json: Complete training history
  - training_plots.png: Visualization of all metrics
  - test_results.json: Final test evaluation
  - checkpoints/: Model checkpoints
  - samples/: Sample generated images

